In [1]:
#Q.12 동시구매 분석과 번들 후보
# 상품 라인(50만 행)에 self-merge를 바로 걸면 메모리 폭발 -> (주문,카테고리)/(주문,상품) 고유쌍으로 축약 후 merge
# support = 동시구매 주문수 / 전체 주문수, lift = P(A,B) / (P(A)*P(B)) : 인기 착시 보정, 1이면 무관, 클수록 진짜 동반구매

In [2]:
import pandas as pd

orders = pd.read_csv("../data/orders.csv", usecols=["order_id", "status"], dtype={"status": "category"})
items = pd.read_csv("../data/order_items.csv", usecols=["order_id", "product_id"])
products = pd.read_csv("../data/products.csv", usecols=["product_id", "product_name", "category"])
products["category"] = products["category"].str.strip()  # 카테고리 앞뒤 공백 오염 정리
products = products.drop_duplicates(subset="product_id")  # product_id 3건 완전 중복 행 정리

df = items.merge(orders, on="order_id", how="left").merge(products, on="product_id", how="left")
df = df[df["status"] != "canceled"]  # 취소 주문은 실제 동시구매가 아니므로 제외
N = df["order_id"].nunique()
print(f"분석 대상 주문수: {N:,}")

분석 대상 주문수: 165,395


In [3]:
def pair_table(df, key):
    uniq = df[["order_id", key]].drop_duplicates()  # 주문x key 고유쌍으로 축약 (self-merge 메모리 안전)
    m = uniq.merge(uniq, on="order_id", suffixes=("_a", "_b"))
    m = m[m[f"{key}_a"] < m[f"{key}_b"]]  # 자기쌍(A==A) 제거 + (A,B)/(B,A) 중복 제거
    pair = m.groupby([f"{key}_a", f"{key}_b"]).size().rename("co_orders").reset_index()

    solo = uniq.groupby(key)["order_id"].nunique()
    pair["support"] = pair["co_orders"] / N
    pair["lift"] = pair["co_orders"] * N / (pair[f"{key}_a"].map(solo) * pair[f"{key}_b"].map(solo))
    return pair.sort_values("co_orders", ascending=False)

In [4]:
# 1. 카테고리 쌍 동시구매 상위표 (원지표 co_orders + 보정지표 support·lift)
cat_pairs = pair_table(df, "category")
cat_pairs.head(10).round(4)

,category_a,category_b,co_orders,support,lift
8,도서,전자,38315,0.2317,0.9166
6,도서,식품,24604,0.1488,0.9168
5,도서,뷰티,24381,0.1474,0.9120
0,가구,도서,23506,0.1421,0.9195
7,도서,의류,23245,0.1405,0.9137
13,식품,전자,21142,0.1278,0.9155
11,뷰티,전자,21084,0.1275,0.9165
4,가구,전자,20197,0.1221,0.9181
14,의류,전자,19990,0.1209,0.9131
9,뷰티,식품,13489,0.0816,0.9133


In [5]:
# 2. 상품 쌍 - 표본이 너무 적은 쌍은 lift가 우연히 튀므로 최소 동시구매 건수로 노이즈 컷
prod_pairs = pair_table(df, "product_id")
MIN_CO = 30
name_map = products.set_index("product_id")["product_name"]
prod_top = prod_pairs[prod_pairs["co_orders"] >= MIN_CO].copy()
prod_top["name_a"] = prod_top["product_id_a"].map(name_map)
prod_top["name_b"] = prod_top["product_id_b"].map(name_map)
prod_top.sort_values("lift", ascending=False).head(10)[["name_a", "name_b", "co_orders", "support", "lift"]].round(4)

,name_a,name_b,co_orders,support,lift
57958,프리미엄 장편소설,코어 이어폰,32,0.0002,1.9927
64652,스탠다드 견과류,코어 향수,32,0.0002,1.9673
16210,데일리 원두커피,코어 의자,39,0.0002,1.7361
49604,스탠다드 홍차,스탠다드 초콜릿,33,0.0002,1.7143
5310,베이직 침대프레임,스탠다드 선반,35,0.0002,1.6646
55094,플러스 초콜릿,프리미엄 립스틱,33,0.0002,1.5661
30815,프리미엄 노트북,스탠다드 에세이,38,0.0002,1.5547
18138,프리미엄 키보드,코어 샴푸,47,0.0003,1.5525
1001,플러스 잡지,데일리 에세이,35,0.0002,1.5247
71653,프리미엄 잡지,플러스 장편소설,40,0.0002,1.4999


### 제출물

- **카테고리 쌍 상위표**(위 셀 1): 원지표(co_orders) 1위는 도서×전자(38,315건, 지지도 23.2%)지만 lift는 0.91~0.92로 전 쌍이 1 미만 — 대형 카테고리끼리라 자주 겹쳐 보일 뿐, 실제로는 우연보다 약한 결합(오히려 약한 음의 연관). **카테고리 수준에서는 "인기 착시"만 있고 진짜 번들 신호는 없다.**
- **상품 번들 후보 3건**(최소 동시구매 30건 이상 중 lift 상위, 위 셀 2):
  1. 프리미엄 장편소설 × 코어 이어폰 — 32건, lift 1.99
  2. 스탠다드 견과류 × 코어 향수 — 32건, lift 1.97
  3. 데일리 원두커피 × 코어 의자 — 39건, lift 1.74
  
  세 쌍 모두 카테고리 lift(~0.92)보다 뚜렷이 높아 우연한 동반구매가 아닌 실제 조합 선호로 해석 가능. 다만 표본이 30~40건대로 작아, 번들 프로모션 전 소규모 A/B 테스트로 재확인 후 확대 적용 권장.